In [1]:
# 데이터 생성 : csv파일을 불러옴
import pandas as pd

df = pd.read_csv('../data/csv/sentiment_data_long.csv')

In [2]:
# 독립, 종속변수 분리
X = df['sentence']
y = df['label']

In [3]:
# 훈련, 테스트 분리 
DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE: ]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE: ]

In [4]:
# 형태소 분리 시 모든 형태소를 포함
# 분리한 단어들을 합침
from konlpy.tag import Okt

def get_preprocessing(sentence):
	okt = Okt()
	result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단, 원형으로
	words = [word for word, pos in result]
	return " ".join(words)

# X_train과 X_test의 있는 문장들을 get_preprocessing을 적용
X_train = X_train.apply(get_preprocessing)
X_test = X_test.apply(get_preprocessing)

In [5]:
# 벡터화 
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode="multi_hot"
)
vectorize_layer.adapt(X_train.tolist())


In [6]:
import tensorflow as tf

# 모델 설계
model = models.Sequential([
	layers.Input((1, ), dtype=tf.string),
	vectorize_layer, 
	layers.Dense(64, activation='relu'),
	layers.Dense(32, activation='relu'),
	layers.Dense(1, activation='sigmoid')#출력층
])


In [7]:
# 모델 설정
model.compile(
	optimizer='adam', 
	loss='binary_crossentropy', 
	metrics=['accuracy']
) 

In [8]:
# 학습 
history = model.fit(
	X_train.values, y_train, epochs=100, verbose=1, validation_split=0.2,
	batch_size=32
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9594 - loss: 0.3498 - val_accuracy: 1.0000 - val_loss: 0.1463
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0440 - val_accuracy: 1.0000 - val_loss: 0.0256
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0057 - val_accuracy: 1.0000 - val_loss: 0.0074
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0021 - val_accuracy: 1.0000 - val_loss: 0.0041
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 9.3030e-04 - val_accuracy: 1.0000 - val_loss: 0.0021
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 6.9672e-04 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 5.4018e-04 - val_accurac

In [9]:
# 평가
_, acc = model.evaluate(X_test.values, y_test)
print(f'정확도 : {acc}')


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 4.8636e-06  
정확도 : 1.0


In [12]:
# 예측
# "너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요"
# 오늘 날이 좋고 기분이 좋아요
# 듣고 있는 음악이 슬퍼서 우울해요
import numpy as np
sentences = [
	"너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요",
	"지루한 장면이 반복되어서 스토리가 너무 개연성이 없고 결말이 허무하기 짝이 없다"
]

predictions = model.predict(np.array(sentences,dtype=object))
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step


array([[0.6088024],
       [0.679114 ]], dtype=float32)